# SPARK AI Model Upgrade — Colab Pro GPU Accelerated Forge

**Project:** SPARK (Smart Protection & Alert Response Kit) — Major Project (IOE Pulchowk / Kathmandu Engineering College)

**Objective:** Upgrade SPARK's 1D CNN fall detection model to achieve clinical targets (**Sensitivity $\ge 90.0\%$**, **Specificity $\ge 90.0\%$**) while strictly satisfying ESP32-S3 microcontroller silicon constraints (**Flash $\le 120\text{ KB}$**, **Internal SRAM Tensor Arena $< 40\text{ KB}$**, **Latency $< 100\text{ ms}$**).

---

### What this Notebook Implements (from `edge-ai-model-forge` Skill & Fleet Research):
1. **Asymmetric Focal Loss:** $\mathcal{L} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$ with $\gamma = 2.0, \alpha = 0.75$ to penalize life-threatening missed falls (False Negatives).
2. **Kinematics-Safe Mixup:** Blends same-activity IMU pairs ($\lambda \ge 0.8$) to regularize feature maps without distorting the $1g$ gravitational reference vector.
3. **Cosine Annealing Learning Rate:** Decays with warm restarts for sharp-minima convergence.
4. **Safety-Constrained & Youden's $J$ Calibration:** Computes the optimal operating cutoff on the held-out validation split rather than defaulting to $p = 0.50$.
5. **Class-Balanced INT8 Quantization:** Enforces a 50/50 Fall vs. Non-Fall calibration distribution to prevent $8g\text{--}16g$ dynamic range clipping.
6. **Silicon Header Export:** Generates 16-byte aligned (`alignas(16)`) C header `spark_cnn_int8.h` ready for ESP-IDF / TFLite Micro compilation.

## 1. Colab Pro GPU Diagnostics & Hardware Acceleration

In [ ]:
!nvidia-smi

import tensorflow as tf
import numpy as np
import os
import sys

gpus = tf.config.list_physical_devices('GPU')
print(f"\n[+] TensorFlow Version: {tf.__version__}")
if gpus:
    print(f"[+] Detected GPU Accelerator: {gpus[0]}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("[!] Running on CPU. For Colab Pro, ensure Runtime -> Change runtime type -> GPU (T4/L4/A100) is enabled.")

## 2. Google Drive Mount & Dataset Loading

Mount your Google Drive to load the prepared SisFall windows (`windows.npy`, `labels.npy`, `meta.csv`).

In [ ]:
from google.colab import drive
import pandas as pd
import os, zipfile, glob, shutil
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Check for dataset folder or uncompressed files directly in Google Drive
LOCAL_DATA_DIR = Path('/content/spark_data')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Search for windows.npy across mounted Google Drive
print('[*] Scanning Google Drive for SPARK dataset files...')
drive_npy_matches = glob.glob('/content/drive/MyDrive/**/windows.npy', recursive=True)

if drive_npy_matches:
    found_dir = Path(drive_npy_matches[0]).parent
    print(f'[+] Detected uncompressed dataset files in Drive: {found_dir}')
    DATA_DIR = found_dir
else:
    # If not found uncompressed, search for the zip bundle and extract to local disk
    zip_matches = glob.glob('/content/drive/MyDrive/**/SPARK_dataset_and_models.zip', recursive=True)
    if zip_matches:
        zip_path = zip_matches[0]
        print(f'[*] Extracting dataset from zip: {zip_path} -> {LOCAL_DATA_DIR}')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(LOCAL_DATA_DIR)
        DATA_DIR = LOCAL_DATA_DIR / 'data' if (LOCAL_DATA_DIR / 'data' / 'windows.npy').exists() else LOCAL_DATA_DIR
    else:
        DATA_DIR = Path('/content/drive/MyDrive/1_A54IjeWyIP6fVuXAUQtyoxYZd5U1RpD')

print(f'\n[+] Final DATA_DIR: {DATA_DIR}')
assert (DATA_DIR / 'windows.npy').exists(), f'Error: windows.npy not found in {DATA_DIR}'
assert (DATA_DIR / 'labels.npy').exists(), f'Error: labels.npy not found in {DATA_DIR}'
assert (DATA_DIR / 'meta.csv').exists(), f'Error: meta.csv not found in {DATA_DIR}'
print('[+] Verified all 3 dataset files exist and are ready for training!')


## 3. Data Ingestion & Subject-Grouped Split (Zero Data Leakage)

Strict 80/10/10 subject-stratified split ensuring no subject appears in both training and test sets.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def load_sisfall(data_dir: Path):
    windows = np.load(data_dir / 'windows.npy').astype(np.float32)
    raw_labels = np.load(data_dir / 'labels.npy')
    
    # Handle both string activity labels ('F01'..'F15' -> 1, 'D01'..'D19' -> 0) and pre-binarized integers
    if raw_labels.dtype.kind in ('U', 'S', 'O'):
        labels_bin = np.where(np.char.startswith(raw_labels.astype(str), 'F'), 1, 0).astype(np.int32)
    else:
        labels_bin = raw_labels.astype(np.int32)
        
    meta = pd.read_csv(data_dir / 'meta.csv')
    subjects = meta['subject'].to_numpy()
    return windows, labels_bin, subjects

def subject_stratified_split(subjects, labels, seed=42):
    gss = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=seed)
    train_idx, temp_idx = next(gss.split(subjects, labels, groups=subjects))
    
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
    sub_val, sub_test = next(gss_val.split(subjects[temp_idx], labels[temp_idx], groups=subjects[temp_idx]))
    
    val_idx = temp_idx[sub_val]
    test_idx = temp_idx[sub_test]
    return train_idx, val_idx, test_idx

print("Loading SisFall windows...")
windows, labels_bin, subjects = load_sisfall(DATA_DIR)
print(f"  Total Windows: {len(windows)}, Window Shape: {windows.shape[1:]}")
print(f"  FALL: {int(labels_bin.sum())}, NON_FALL: {int((1 - labels_bin).sum())}")

train_idx, val_idx, test_idx = subject_stratified_split(subjects, labels_bin)
X_train, y_train = windows[train_idx], labels_bin[train_idx]
X_val, y_val = windows[val_idx], labels_bin[val_idx]
X_test, y_test = windows[test_idx], labels_bin[test_idx]

print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


## 4. Kinematics-Safe Augmentation (Jitter & Mixup)

Augments time-series while preserving the $1g$ downward gravity alignment and angular velocity consistency.

In [ ]:
def augment_windows(windows: np.ndarray, rng: np.random.Generator, max_shift=5, scale_range=0.05, noise_std=0.01):
    aug = windows.copy()
    N, T, C = windows.shape
    for i in range(N):
        shift = rng.integers(-max_shift, max_shift + 1)
        if shift != 0:
            aug[i] = np.roll(aug[i], shift, axis=0)
        scale = 1.0 + rng.uniform(-scale_range, scale_range, size=(1, C))
        aug[i] *= scale
        noise = rng.normal(0.0, noise_std, size=(T, C))
        aug[i] += noise
    return aug.astype(np.float32)

def mixup_augment_windows(windows: np.ndarray, labels: np.ndarray, rng: np.random.Generator, alpha=0.2):
    mixed_windows = windows.copy()
    mixed_labels = labels.copy()
    for cls in [0, 1]:
        cls_idx = np.where(labels == cls)[0]
        if len(cls_idx) < 2:
            continue
        shuffled = rng.permutation(cls_idx)
        lam = rng.beta(alpha, alpha, size=len(cls_idx))
        lam = np.maximum(lam, 1.0 - lam) # Constrained to [0.75, 1.0] for kinematic stability
        for i, idx in enumerate(cls_idx):
            partner = shuffled[i]
            l = float(lam[i])
            mixed_windows[idx] = l * windows[idx] + (1.0 - l) * windows[partner]
    return mixed_windows.astype(np.float32), mixed_labels

rng = np.random.default_rng(42)
X_train_aug = augment_windows(X_train, rng)
X_train_combined = np.concatenate([X_train, X_train_aug], axis=0)
y_train_combined = np.concatenate([y_train, y_train], axis=0)

# Apply Mixup
X_mix, y_mix = mixup_augment_windows(X_train_combined, y_train_combined, rng, alpha=0.2)
X_train_fit = np.concatenate([X_train_combined, X_mix], axis=0)
y_train_fit = np.concatenate([y_train_combined, y_mix], axis=0)
shuf = rng.permutation(len(y_train_fit))
X_train_fit, y_train_fit = X_train_fit[shuf], y_train_fit[shuf]

print(f"[*] Final augmented training set: {len(X_train_fit)} windows")

## 5. Model Architecture & Asymmetric Focal Loss

Constructs the proposal-locked 1D CNN with **Asymmetric Focal Loss** ($\gamma=2.0, \alpha=0.75$) and Cosine Annealing.

In [ ]:
class AsymmetricFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, name="asymmetric_focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.squeeze(y_true), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(y_true, depth=2)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        alpha_t = tf.where(tf.equal(y_true, 1), self.alpha, 1.0 - self.alpha)
        loss = -alpha_t * focal_weight * tf.math.log(p_t)
        return tf.reduce_mean(loss)

def build_spark_cnn(input_shape=(200, 6), dropout_rate=0.20, use_batch_norm=False, loss_type='focal', lr=1e-3, steps_per_epoch=100):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv1D(32, kernel_size=5, activation='relu')(inputs)
    if use_batch_norm:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool1D(pool_size=2)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    x = tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu')(x)
    if use_batch_norm:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(2, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(initial_learning_rate=lr, first_decay_steps=max(10*steps_per_epoch, 100), alpha=1e-5)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    if loss_type == 'focal':
        loss = AsymmetricFocalLoss(gamma=2.0, alpha=0.75)
    else:
        loss = 'sparse_categorical_crossentropy'

    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
    return model

steps_per_epoch = len(X_train_fit) // 64
model = build_spark_cnn(input_shape=X_train.shape[1:], loss_type='focal', steps_per_epoch=steps_per_epoch)
model.summary()

## 6. GPU-Accelerated Training on Colab Pro

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("[*] Training Upgraded SPARK 1D CNN with Asymmetric Focal Loss on GPU...")
history = model.fit(
    X_train_fit, y_train_fit,
    validation_data=(X_val, y_val),
    batch_size=64,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

## 7. Decision Threshold Calibration (Youden's $J$ & Safety Constraint)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, f1_score

val_probs = model.predict(X_val)[:, 1]
fpr, tpr, thresholds = roc_curve(y_val, val_probs)

# 1. Youden's J Index
j_scores = tpr - fpr
optimal_threshold = float(thresholds[np.argmax(j_scores)])

# 2. Safety-Constrained Threshold (Sensitivity >= 90%)
valid_idx = np.where(tpr >= 0.90)[0]
if len(valid_idx) > 0:
    best_idx = int(np.argmin(fpr[valid_idx]))
    safety_threshold = float(thresholds[valid_idx[best_idx]])
else:
    safety_threshold = optimal_threshold

print(f"[*] Default Threshold:            0.5000")
print(f"[*] Youden's J Threshold:         {optimal_threshold:.4f}")
print(f"[*] Safety-Constrained Threshold: {safety_threshold:.4f} (Sens >= 90%)")

## 8. Held-Out Test Set Evaluation

In [ ]:
def evaluate_model(model, X, y, threshold=0.50):
    y_prob = model.predict(X, verbose=0)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(((y_pred == 1) & (y == 1)).sum())
    fn = int(((y_pred == 0) & (y == 1)).sum())
    tn = int(((y_pred == 0) & (y == 0)).sum())
    fp = int(((y_pred == 1) & (y == 0)).sum())
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = float(f1_score(y, y_pred, zero_division=0))
    auc = float(roc_auc_score(y, y_prob))
    return {'sensitivity': sens, 'specificity': spec, 'f1': f1, 'auc_roc': auc}

m_default = evaluate_model(model, X_test, y_test, threshold=0.50)
m_youden = evaluate_model(model, X_test, y_test, threshold=optimal_threshold)
m_safety = evaluate_model(model, X_test, y_test, threshold=safety_threshold)

print("=" * 75)
print("HELD-OUT TEST SET EVALUATION MATRIX")
print("=" * 75)
print(f"{'Metric':<25} {'Default (0.50)':>15} {'Youden (' + f'{optimal_threshold:.2f}' + ')':>15} {'Safety (' + f'{safety_threshold:.2f}' + ')':>15}")
print("-" * 75)
print(f"{'Sensitivity (FALL)':<25} {m_default['sensitivity']:>15.4f} {m_youden['sensitivity']:>15.4f} {m_safety['sensitivity']:>15.4f}  [Target >= 0.90]")
print(f"{'Specificity (NON_FALL)':<25} {m_default['specificity']:>15.4f} {m_youden['specificity']:>15.4f} {m_safety['specificity']:>15.4f}  [Target >= 0.90]")
print(f"{'F1-Score':<25} {m_default['f1']:>15.4f} {m_youden['f1']:>15.4f} {m_safety['f1']:>15.4f}")
print(f"{'AUC-ROC':<25} {m_default['auc_roc']:>15.4f} {m_youden['auc_roc']:>15.4f} {m_safety['auc_roc']:>15.4f}")
print("=" * 75)

## 9. Class-Balanced Post-Training Quantization (INT8) & Silicon Export

Enforces a strict 50% Fall / 50% Non-Fall calibration distribution to preserve $8g\text{--}16g$ impact dynamic ranges, then exports the 16-byte aligned C header `spark_cnn_int8.h`.

In [ ]:
def make_balanced_representative_dataset(windows, labels, n_samples=300):
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    n_half = n_samples // 2
    chosen_pos = np.random.choice(pos_idx, size=min(n_half, len(pos_idx)), replace=False)
    chosen_neg = np.random.choice(neg_idx, size=min(n_half, len(neg_idx)), replace=False)
    calib_idx = np.concatenate([chosen_pos, chosen_neg])
    np.random.shuffle(calib_idx)
    calib_data = windows[calib_idx].astype(np.float32)
    def generator():
        for i in range(len(calib_data)):
            yield [calib_data[i:i+1]]
    return generator

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = make_balanced_representative_dataset(X_train, y_train, n_samples=300)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()
model_size_kb = len(tflite_quant_model) / 1024.0

print(f"[+] Quantized INT8 Model Size: {len(tflite_quant_model)} bytes ({model_size_kb:.2f} KB)")
print(f"[+] Satisfies ESP32-S3 Flash Budget (<= 120 KB): {model_size_kb <= 120.0}")

# Export Aligned C Header
hex_bytes = [f"0x{b:02x}" for b in tflite_quant_model]
lines = []
for i in range(0, len(hex_bytes), 12):
    lines.append("  " + ", ".join(hex_bytes[i:i+12]))

header_content = f"""// Auto-generated SPARK INT8 Model Header for ESP32-S3 (esp-nn)
// Flash Size: {len(tflite_quant_model)} bytes ({model_size_kb:.2f} KB)
// Optimal Operating Threshold: {safety_threshold:.4f}

#pragma once
#include <stdalign.h>

alignas(16) const unsigned char spark_cnn_model[] = {{\n{",\n".join(lines)}\n}};

const unsigned int spark_cnn_model_len = {len(tflite_quant_model)};
const float spark_cnn_optimal_threshold = {safety_threshold:.4f}f;
"""

with open('spark_cnn_int8.h', 'w') as f:
    f.write(header_content)

print("[+] Generated 'spark_cnn_int8.h' ready for firmware embedding!")
# Optional: Save directly back to your mounted Google Drive folder
drive_save_candidates = glob.glob('/content/drive/MyDrive/**/1_A54IjeWyIP6fVuXAUQtyoxYZd5U1RpD*', recursive=True)
if drive_save_candidates:
    save_target = Path(drive_save_candidates[0])
    import shutil
    shutil.copy('spark_cnn_int8.h', save_target / 'spark_cnn_int8_upgraded.h')
    with open(save_target / 'spark_cnn_int8_upgraded.tflite', 'wb') as f:
        f.write(tflite_quant_model)
    print(f'[+] Successfully saved upgraded models directly back to Google Drive: {save_target}')
else:
    print('[*] You can download spark_cnn_int8.h directly from Colab files pane.')


## 10. Compute Unit Conservation & Programmatic Runtime Teardown

Automatically unassigns the Colab GPU runtime to conserve Colab Pro compute units (`compute units / credits`) as soon as training, quantization, and artifact persistence to Google Drive complete.

In [ ]:
# Auto-disconnect runtime to prevent draining Colab Pro compute units
print("[+] All training, evaluation, and Drive exports completed successfully!")
print("[*] Programmatically unassigning runtime to conserve Colab compute credits...")

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"[*] Note: runtime.unassign() called (or running outside Colab): {e}")
